# Minimizer density — random (k, w) pairs

## Setup

### Imports

In [1]:
import os
import sys
import random
import hashlib
import math
import cmath
import numpy as np
import pandas as pd
from collections import deque
from decimal import Decimal, getcontext
from tqdm import tqdm
from joblib import Parallel, delayed
getcontext().prec = 100

# Print Python and package versions
print(f"Python: {sys.version}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
import importlib.metadata
print(f"tqdm: {importlib.metadata.version('tqdm')}")
print(f"joblib: {importlib.metadata.version('joblib')}")


Python: 3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]
numpy: 1.26.4
pandas: 2.3.0
tqdm: 4.67.1
joblib: 1.3.2


### Measure density

In [2]:
# ----- Tiebreaking: get_kmer_id_dna (used by density) -----
def hash_kmer(k, kmer, seed):
    kmer_seed_sum = kmer + seed
    kmer_as_bytes = kmer_seed_sum.to_bytes(2 * k, byteorder='big')
    hashed_kmer = hashlib.sha256(kmer_as_bytes).digest()
    return int.from_bytes(hashed_kmer, byteorder='big')


def get_kmer_id_dna(k, kmer, seed):
    hashed_kmer = hash_kmer(k, kmer, seed) % (4 ** k)
    return (4 ** k) * hashed_kmer + kmer


# ----- Density sampling (DNA, includes tiebreaking) -----
# Two-phase: (1) build k-mers from sequence and assign ranks to all; (2) sliding window over rank array.
def random_density_function_dna_includes_tiebreaking(w, k, function, n_samples=10000, seed=1):
    """Estimate density: assign ranks to all k-mers in sequence, then slide over rank array to count sampled positions."""
    k_mask = (1 << (2 * k)) - 1
    total_nucleotides = n_samples + w + k - 1
    rng = np.random.default_rng(seed)
    nucleotides = rng.integers(0, 4, size=total_nucleotides, dtype=np.uint32)

    # Phase 1: build array of all k-mers (Python ints)
    n_kmers = total_nucleotides - k + 1
    kmers = []
    kmer = 0
    for pos in range(k):
        kmer = (kmer << 2) | int(nucleotides[pos])
    kmers.append(kmer & k_mask)
    for i in range(1, n_kmers):
        kmer = ((kmer << 2) | int(nucleotides[i + k - 1])) & k_mask
        kmers.append(kmer)

    # Phase 2: assign rank to every k-mer (rank = minimizer_rank << 8k + tiebreak_id)
    shift_8k = 8 * k
    ranks = [
        (function(km, w, k, seed) << shift_8k) + get_kmer_id_dna(k, km, seed)
        for km in kmers
    ]

    # Phase 3: sliding window over rank array; count when minimizer position changes
    sampled_positions = 0
    last_sampled = -1
    for t in range(n_samples):
        window = ranks[t : t + w]
        min_pos = t + window.index(min(window))
        if min_pos != last_sampled:
            sampled_positions += 1
            last_sampled = min_pos

    return sampled_positions / n_kmers  # density = minimizer positions / all k-mers

In [3]:
# ----- CPU count: prefer physical cores for CPU-bound parallel work -----
def _physical_cpu_count():
    try:
        import psutil
        return psutil.cpu_count(logical=False) or os.cpu_count() or 1
    except Exception:
        return os.cpu_count() or 1


def _effective_n_jobs(n_jobs):
    """-1 => physical core count; 1 => 1; else use as-is."""
    if n_jobs == -1:
        return _physical_cpu_count()
    return n_jobs


# ----- Helpers for scheme vs raw function (used by sampling) -----
def _scheme_name(scheme):
    """Display name for a scheme (MinimizerScheme instance or raw function)."""
    return scheme.name if hasattr(scheme, "name") else scheme.__name__


def _scheme_minimizer_for_repeat(scheme, repeat_seed):
    """Return minimizer function for this repeat. OC/Miniception use repeat_seed for inner order."""
    if hasattr(scheme, "get_minimizer"):
        return scheme.get_minimizer(seed=repeat_seed)
    return scheme


# ----- Sampling: raw density per repeat -----
def _run_one_density(w, k, minimizer_func, n_samples, seed):
    """Single density run (for parallel dispatch)."""
    return random_density_function_dna_includes_tiebreaking(w, k, minimizer_func, n_samples, seed=seed)


def _compute_row_repeats(w, k, schemes, n_samples, repeats, n_jobs_inner):
    """One (w, k): density factor per repeat for each scheme."""
    row = {"w": w, "k": k}
    scheme_vals = {}
    for scheme in schemes:
        name = _scheme_name(scheme)
        tasks = [
            delayed(_run_one_density)(w, k, _scheme_minimizer_for_repeat(scheme, i), n_samples, i)
            for i in range(repeats)
        ]
        densities = Parallel(n_jobs=_effective_n_jobs(n_jobs_inner))(tasks)
        scheme_vals[name] = np.asarray(densities, dtype=float) * (w + 1)
    row["_scheme_vals"] = scheme_vals
    return row


def _rows_to_single_wide_df(rows, schemes):
    """One CSV: columns k, w, then for each scheme: {name}_mean, {name}_std, {name}_min, {name}_max."""
    names = [_scheme_name(s) for s in schemes]
    out_rows = []
    for r in rows:
        d = {"k": r["k"], "w": r["w"]}
        for name in names:
            v = np.asarray(r["_scheme_vals"][name], dtype=float)
            d[f"{name}_mean"] = float(np.mean(v))
            d[f"{name}_std"] = float(np.std(v, ddof=1)) if v.size > 1 else 0.0
            d[f"{name}_min"] = float(np.min(v))
            d[f"{name}_max"] = float(np.max(v))
        out_rows.append(d)
    return pd.DataFrame(out_rows)


def sample_density_random_pairs(pairs, schemes, n_samples=10000, repeats=5, n_jobs=-1):
    """pairs: iterable of (w, k). Returns one DataFrame (density factors across repeats)."""
    pair_list = list(pairs)
    if n_jobs == 1:
        rows = [
            _compute_row_repeats(w, k, schemes, n_samples, repeats, n_jobs_inner=-1)
            for w, k in tqdm(pair_list, desc="random (w,k) pairs", unit="pair")
        ]
    else:
        rows = Parallel(n_jobs=_effective_n_jobs(n_jobs))(
            delayed(_compute_row_repeats)(w, k, schemes, n_samples, repeats, n_jobs_inner=1)
            for w, k in tqdm(pair_list, desc="random (w,k) pairs", unit="pair")
        )
    return _rows_to_single_wide_df(rows, schemes)


## Minimizers

### Double Decycling

In [4]:
# ----- Double decycling (DNA) -----
_DD_BASE_WEIGHTS = (65, 67, 71, 84)  # C++ style: use number as-is (ASCII) - we used these weights in GreedyMini
# _DD_BASE_WEIGHTS = (0, 1, 2, 3)   # Default
_decycling_roots_cache = {}


def _get_decycling_roots(k):
    if k not in _decycling_roots_cache:
        _decycling_roots_cache[k] = [cmath.exp(2 * cmath.pi * 1j * i / k) for i in range(k)]
    return _decycling_roots_cache[k]


_DEPS = 1e-12


def get_decycling_set_dna(kmer, w, k):
    roots = _get_decycling_roots(k)
    x = sum(roots[i] * _DD_BASE_WEIGHTS[(kmer >> (2 * (k - 1 - i))) & 0b11] for i in range(k))
    mag = abs(x)
    if mag <= _DEPS:
        return 2  # Neither (arg undefined at origin; PMC "all rotations I(x)=0" edge case)
    angle = cmath.phase(x)
    two_pi_k = 2 * math.pi / k
    lo_pos = math.pi - two_pi_k - _DEPS
    lo_neg = -two_pi_k - _DEPS
    hi_neg = 0 + _DEPS
    if lo_pos <= angle <= math.pi + _DEPS:
        return 0  # Decycling-positive (Dk)
    if lo_neg <= angle <= hi_neg:
        return 1  # Decycling-negative (D~k)
    return 2  # Neither


def double_decycling_dna(kmer, w, k, seed=None):
    return get_decycling_set_dna(kmer, w, k)

### Miniception and open-closed

In [5]:
# Syncmer helpers: s-mer at position i (0..k-s) in DNA k-mer (2 bits per base)
def _smer_at(kmer, k, s, i):
    """Extract s-mer at start position i (0 = leftmost)."""
    shift = 2 * (k - s - i)
    mask = (1 << (2 * s)) - 1
    return (kmer >> shift) & mask


def _syncmer_class_dna(kmer, k, s):
    """
    Return (is_open, is_closed) for a k-mer.
    Closed: smallest s-mer at position 0 or k-s.
    Open: smallest s-mer at position v = (k-s)//2.
    """
    n_pos = k - s + 1
    if n_pos <= 0:
        return False, False
    smers = [(_smer_at(kmer, k, s, i), i) for i in range(n_pos)]
    min_val = min(s[0] for s in smers)
    min_positions = [i for v, i in smers if v == min_val]
    argmin = min_positions[0]  # leftmost min for tiebreak
    v_mid = (k - s) // 2
    is_closed = argmin == 0 or argmin == k - s
    is_open = argmin == v_mid
    return is_open, is_closed


def _syncmer_param_s(k, default_s=4):
    """Syncmer length s: min(default_s, k), at least 1."""
    return max(1, min(default_s, k))


def miniception_dna(kmer, w, k, seed=None):
    """
    Miniception: smallest closed syncmer in window; if none, smallest k-mer.
    Inner order (within class) uses seed so it is random between runs, constant within run.
    """
    s = _syncmer_param_s(k)
    is_open, is_closed = _syncmer_class_dna(kmer, k, s)
    run_seed = seed if seed is not None else 0
    inner_id = get_kmer_id_dna(k, kmer, run_seed)
    L = 1 << (8 * k)
    if is_closed:
        return 0 * L + inner_id
    return 1 * L + inner_id


def open_closed_dna(kmer, w, k, seed=None):
    """
    Open-closed (OC) minimizer: prefer open syncmer > closed syncmer > other.
    Inner order (within each class) uses seed: random between runs, constant within run.
    """
    s = _syncmer_param_s(k)
    is_open, is_closed = _syncmer_class_dna(kmer, k, s)
    run_seed = seed if seed is not None else 0
    inner_id = get_kmer_id_dna(k, kmer, run_seed)
    L = 1 << (8 * k)
    if is_open:
        return 0 * L + inner_id
    if is_closed:
        return 1 * L + inner_id
    return 2 * L + inner_id

### One-zero

In [6]:
# ----- One-zero (DNA) -----
def _lemma_1_order_rank(projection, k):
    order_list = []
    order_list.append((1 << k) - 2)
    order_list.append((1 << (k - 1)) - 2)
    for i in range(2, k - 1):
        order_list.append(2 * ((1 << (k - 1 - i)) - 1))
    order_list.append(1)
    order_list.append((1 << k) - 1)
    order_list.append(0)
    order_list = list(dict.fromkeys(order_list))
    rank_map = {p: r for r, p in enumerate(order_list)}
    return rank_map.get(projection, -1)


def _build_lemma_1_set(k):
    lemma_1_projs = {(1 << k) - 1, 0, 1, (1 << k) - 2}
    for i in range(2, k):
        ones_count = k - i
        pattern = ((1 << ones_count) - 1) << 1
        lemma_1_projs.add(pattern)
    return lemma_1_projs




def one_zero_minimizer_key(k, kmer):
    """Variant: tail/head boundary at first subsequent '10'. Projection = high bit per base (G/T=1, A/C=0), matching 4-ary 10-order."""
    projection = 0
    for i in range(k):
        nuc = (kmer >> (2 * (k - 1 - i))) & 0b11
        bit = 1 if nuc == 3 else 0
        projection = (projection << 1) | bit

    def get_bit(val, pos):
        return (val >> (k - 1 - pos)) & 1

    # Require leading '10' as in one_zero_minimizer_key
    if get_bit(projection, 0) == 1 and get_bit(projection, 1) == 0:
        head_len = 2
        prev = 0  # bit 1 is 0 here
        for i in range(2, k):
            curr = get_bit(projection, i)
            head_len += 1
            if prev == 1 and curr == 0:
                break
            prev = curr
        # if last bit is 1 then reduce head_len by 1 to account for 10000000000000000000001
        if (prev == 1):
            head_len -= 1

        tail_len = k - head_len
        tail_val = projection & ((1 << tail_len) - 1)
        return (0, head_len, tail_val)
        

    lemma_1_projs = _build_lemma_1_set(k)
    if projection in lemma_1_projs:
        return (1, _lemma_1_order_rank(projection, k))
    return (2, projection)


def one_zero_minimizer_rank(kmer, k):
    t = one_zero_minimizer_key(k, kmer)
    L = (1 << (2 * k)) + 1
    if t[0] == 0:
        head_len, tail_val = t[1], t[2]
        return 0 * (L ** 3) - (head_len) * (L ** 2) - (tail_val) * L + kmer
    elif t[0] == 1:
        rank_in_lemma_1 = t[1]
        return 1 * (L ** 3) + rank_in_lemma_1 * (L ** 2) + kmer
    else:
        return 2 * (L ** 3) - kmer


def one_zero_v2_dna(kmer, w, k, seed=None):
    return one_zero_minimizer_rank(kmer, k)

### ABB+

In [7]:
def abb_plus_score_dna(kmer, k):
    """Pos 0 lex (A<C<T<G); pos 1..k-1 non-A < A; tiebreak lex."""
    nuc0 = (kmer >> (2 * (k - 1))) & 0b11
    key = nuc0
    for i in range(1, k):
        nuc = (kmer >> (2 * (k - 1 - i))) & 0b11
        key = key * 2 + (1 if nuc == 0 else 0)
    return key * (1 << (2 * k)) + kmer


def abb_plus_dna(kmer, w, k, seed=None):
    """Minimizer using ABB+ rank only (no shift)."""
    return abb_plus_score_dna(kmer, k)

### Wrappers

In [8]:
class MinimizerScheme:
    """Base: .name and .get_minimizer(seed=None) -> (kmer, w, k, seed=None) -> rank."""

    name = "base"

    def get_minimizer(self, seed=None):
        raise NotImplementedError


class DoubleDecyclingScheme(MinimizerScheme):
    name = "double_decycling_dna"

    def get_minimizer(self, seed=None):
        def _fn(kmer, w, k, seed=None):
            return get_decycling_set_dna(kmer, w, k)
        return _fn


class MiniceptionScheme(MinimizerScheme):
    """Uses seed in get_minimizer(seed) for inner random order per run."""

    name = "miniception_dna"

    def get_minimizer(self, seed=None):
        run_seed = seed if seed is not None else 0

        def _fn(kmer, w, k, seed=None):
            return miniception_dna(kmer, w, k, run_seed)
        return _fn


class OpenClosedScheme(MinimizerScheme):
    """Uses seed in get_minimizer(seed) for inner random order per run."""

    name = "open_closed_dna"

    def get_minimizer(self, seed=None):
        run_seed = seed if seed is not None else 0

        def _fn(kmer, w, k, seed=None):
            return open_closed_dna(kmer, w, k, run_seed)
        return _fn


class OneZeroScheme(MinimizerScheme):
    name = "one_zero_dna"

    def get_minimizer(self, seed=None):
        def _fn(kmer, w, k, seed=None):
            return one_zero_minimizer_rank(kmer, k)
        return _fn



class ABBPlusScheme(MinimizerScheme):
    name = "abb_plus_dna"

    def get_minimizer(self, seed=None):
        def _fn(kmer, w, k, seed=None):
            return abb_plus_score_dna(kmer, k)
        return _fn

## Measure Density

In [9]:
# All DNA minimizers as schemes (name + get_minimizer(seed)); OC and Miniception use seed per run.
SCHEMES_DNA = [
    DoubleDecyclingScheme(),
    OneZeroScheme(),
    ABBPlusScheme(),
    MiniceptionScheme(),
    OpenClosedScheme(),
]
FUNCTIONS_DNA = SCHEMES_DNA  # alias for sampling/plots


## Config and run


In [10]:
N_SAMPLES = 2_000_000
REPEATS = 5
OUTPUT_DIR = "output"

N_RANDOM_PAIRS = 100
PAIR_SEED = 42

K_MIN, K_MAX = 6, 32
W_MAX = 100


In [11]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

rng = np.random.default_rng(PAIR_SEED)

# Note: pairs are (w, k)
all_pairs = [
    (w, k)
    for k in range(K_MIN, K_MAX + 1)
    for w in range(k - 2, W_MAX + 1)
]
if N_RANDOM_PAIRS > len(all_pairs):
    raise ValueError(f"N_RANDOM_PAIRS={N_RANDOM_PAIRS} exceeds {len(all_pairs)} distinct pairs")
idx = rng.choice(len(all_pairs), size=N_RANDOM_PAIRS, replace=False)
pairs = [all_pairs[i] for i in idx]

df = sample_density_random_pairs(pairs, FUNCTIONS_DNA, n_samples=N_SAMPLES, repeats=REPEATS)
path = os.path.join(OUTPUT_DIR, "density_random_pairs_stats.csv")
df.to_csv(path, index=False)
print("Saved", path, "rows", len(df))


random (w,k) pairs: 100%|██████████| 100/100 [54:53<00:00, 32.94s/pair]


Saved output\density_random_pairs_stats.csv rows 100
